<a href="https://colab.research.google.com/github/almudenaandradesoto/almudenaandradesoto-DataScience-GenAI-Submissions/blob/main/Assignment_6/6_02_Dense_Neural_Network_with_PyTorch_COMPLETED.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=1xqQczl0FG-qtNA2_WQYuWePW9oU8irqJ)

# 6.02 Dense Neural Network (with PyTorch)
This will expand on our logistic regression example and take us through building our first neural network. If you haven't already, be sure to check (and if neccessary) switch to GPU processing by clicking Runtime > Change runtime type and selecting GPU. We can test this has worked with the following code:

In [1]:
import torch

# Check for GPU availability
print("Num GPUs Available: ", torch.cuda.device_count())

Num GPUs Available:  1


Hopefully your code shows you have 1 GPU available! Next let's get some data. We'll start with another in-built dataset:

In [2]:
# upload an in-built Python (OK semi-in-built) dataset
from sklearn.datasets import load_diabetes

import pandas as pd
import numpy as np

# import the data
data = load_diabetes()
data

{'data': array([[ 0.03807591,  0.05068012,  0.06169621, ..., -0.00259226,
          0.01990749, -0.01764613],
        [-0.00188202, -0.04464164, -0.05147406, ..., -0.03949338,
         -0.06833155, -0.09220405],
        [ 0.08529891,  0.05068012,  0.04445121, ..., -0.00259226,
          0.00286131, -0.02593034],
        ...,
        [ 0.04170844,  0.05068012, -0.01590626, ..., -0.01107952,
         -0.04688253,  0.01549073],
        [-0.04547248, -0.04464164,  0.03906215, ...,  0.02655962,
          0.04452873, -0.02593034],
        [-0.04547248, -0.04464164, -0.0730303 , ..., -0.03949338,
         -0.00422151,  0.00306441]], shape=(442, 10)),
 'target': array([151.,  75., 141., 206., 135.,  97., 138.,  63., 110., 310., 101.,
         69., 179., 185., 118., 171., 166., 144.,  97., 168.,  68.,  49.,
         68., 245., 184., 202., 137.,  85., 131., 283., 129.,  59., 341.,
         87.,  65., 102., 265., 276., 252.,  90., 100.,  55.,  61.,  92.,
        259.,  53., 190., 142.,  75., 142.

We are working on a regression problem, with "structured" data which has already been cleaned and normalised. We can skip the usual cleaning/engineering steps. However, we do need to get the data into PyTorch:

In [3]:
# Convert data to PyTorch tensors
X = torch.tensor(data.data, dtype=torch.float32)
y = torch.tensor(data.target, dtype=torch.float32).reshape(-1, 1) # Reshape y to be a column vector

Now our data is stored in tensors we can do train/test splitting as before (in fact we can use sklearn as before):

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

torch.Size([353, 10]) torch.Size([353, 1])
torch.Size([89, 10]) torch.Size([89, 1])


Now we can set up our batches for training. As we have a nice round 400 let's go with batches of 50 (8 batches in total). We'll also seperate the features and labels:

In [5]:
from torch.utils.data import TensorDataset, DataLoader

# Create TensorDatasets and DataLoaders
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=50, shuffle=True)

test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=50, shuffle=False)

Now its time to build our model. We'll keep it simple ... a model with an input layer of 10 features and then 2x _Dense_ (fully connected) layers each with 5 neurons and ReLU activation. Our output layer will be size=1 given this is a regression problem and we want a single value output per prediction.

This will be easier to understand if you have read through the logistic regression tutorial.

In [6]:
import torch
import torch.nn as nn

# Define the model
class DiabetesModel(nn.Module):
    def __init__(self):
        super(DiabetesModel, self).__init__()
        # we'll set up the layers as a sequence using nn.Sequential
        self.layers = nn.Sequential(

            # first layer will be a linear layer that has 5x neurons
            # (5x sets of linear regression)
            # the layer takes the 10 features as input (i.e. 10, 5)
            nn.Linear(10, 5),

            nn.ReLU(), # ReLU activation

            # second linear layer again has 5 neurons
            # this time taking the input as the output of the last layer
            # (which had 5x neurons)
            nn.Linear(5, 5),

            nn.ReLU(), # ReLU again

            # last linear layer takes the output from the previous 5 neurons
            # this time its a single output with no activation
            # i.e. this is the predicitons (regression)
            nn.Linear(5, 1)
        )

    def forward(self, x):
        return self.layers(x) # pass the data through the layers

As before we need to create a model object, specify the loss (criterion) and an optimiser (which we cover next week):

In [7]:
import torch.optim as optim

# Initialize the model, loss function, and optimizer
model = DiabetesModel()
criterion = nn.MSELoss() # MSE loss function
optimiser = optim.Adam(model.parameters(), lr=0.001)

Now we can train the model. Again, the logistic regression tutorial (6.01) may help you undertstand this:

In [8]:
# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Training loop (example - you'll likely want to add more epochs)
epochs = 100 # 100 epochs

for epoch in range(epochs):
  # use the train_loader to pass the inputs (x) and targets (y)
  for inputs, targets in train_loader:
    # pass to the GPU (hopefully)
    inputs, targets = inputs.to(device), targets.to(device)

    # pass model to GPU as well
    model.to(device)

    model.train() # put the model object in train mode
    optimiser.zero_grad() # reset the gradiants
    outputs = model(inputs) # create outputs
    loss = criterion(outputs, targets) # compare with Y to get loss
    loss.backward() # backpropogate the loss (next week)
    optimiser.step() # # update the parameters based on this round of training

  # every 10 steps we will print out the current loss
    if (epoch+1) % 10 == 0: # modular arithmetic
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {round(loss.item(), 4)}')

Epoch [10/100], Loss: 29869.2559
Epoch [10/100], Loss: 32171.4414
Epoch [10/100], Loss: 30283.6387
Epoch [10/100], Loss: 29672.8242
Epoch [10/100], Loss: 31722.3027
Epoch [10/100], Loss: 29254.1602
Epoch [10/100], Loss: 27048.9336
Epoch [10/100], Loss: 10869.9141
Epoch [20/100], Loss: 28050.7988
Epoch [20/100], Loss: 26167.3203
Epoch [20/100], Loss: 30195.9648
Epoch [20/100], Loss: 31825.2168
Epoch [20/100], Loss: 37628.5781
Epoch [20/100], Loss: 23153.5176
Epoch [20/100], Loss: 31794.8223
Epoch [20/100], Loss: 26067.3008
Epoch [30/100], Loss: 27660.9551
Epoch [30/100], Loss: 30890.6895
Epoch [30/100], Loss: 31034.4746
Epoch [30/100], Loss: 24906.8652
Epoch [30/100], Loss: 27250.8594
Epoch [30/100], Loss: 34995.0078
Epoch [30/100], Loss: 31436.3691
Epoch [30/100], Loss: 33688.2344
Epoch [40/100], Loss: 30110.6895
Epoch [40/100], Loss: 28523.7461
Epoch [40/100], Loss: 28805.7695
Epoch [40/100], Loss: 29558.7148
Epoch [40/100], Loss: 24679.8203
Epoch [40/100], Loss: 32613.1113
Epoch [40/

We can see loss is significantly lower at the end than it was at the start. However, it is also bouncing around a little still which suggests the model needs more training (100 epochs is not a lot in deep learning terms). However, let's evaluate as before:

In [9]:
# Evaluation (example)
model.eval() # testing mode
mse_values = [] # collect the MSE scores

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs) # predict the test data

        # Calculate Mean Squared Error
        mse = criterion(outputs, targets) # calcualte mse for the batch
        mse_values.append(mse.item()) # add to the list of MSE values

# Calculate and print the average MSE
avg_mse = np.mean(mse_values)
print(f"Average MSE on test set: {avg_mse}")

Average MSE on test set: 26079.509765625


MSE looks expected given training (no obvious sign of overfitting). However, we probably can get better results with tuning and more epochs.

Let's run the loop again a little differently to collect the predicted values (y_hat) and actuals (y) and add them to a dataset for comparions:

In [10]:
# Evaluation
model.eval()
predictions = []
actuals = []

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        predictions.extend(outputs.cpu().numpy())
        actuals.extend(targets.cpu().numpy())

# Create DataFrame
results_df = pd.DataFrame({'Predicted': np.array(predictions).flatten(), 'Actual': np.array(actuals).flatten()})
results_df

,Predicted,Actual
0,0.361152,219.0
1,0.361152,70.0
2,0.361152,202.0
3,0.361152,230.0
4,0.361152,111.0
...,...,...
84,0.361152,153.0
85,0.361152,98.0
86,0.361152,37.0
87,0.361152,63.0


Side-by-side, they don't look great. Can you improve them?

<br><br>

## EXERCISE #1
Try increasing the number of epochs to 1,000 (when the model is fairly well trained then the results printed for each 10x epochs will be fairly stable and not change much). Does this give better results?

<br><br>

## EXERCISE #2 (optional)
Try experimenting with the architecture (number of neurons and/or number of layers). Can we reach an optimal architecture?

In [11]:
### Exercise #1 – Training the Neural Network for 1,000 Epochs

import torch
import torch.nn as nn

# Define the model
class DiabetesModel1000(nn.Module):
    def __init__(self):
        super(DiabetesModel1000, self).__init__()

        # we'll set up the layers as a sequence using nn.Sequential
        self.layers = nn.Sequential(

            # first layer takes 10 features as input and has 5 neurons
            nn.Linear(10, 5),

            nn.ReLU(), # ReLU activation

            # second layer takes 5 inputs and has 5 neurons
            nn.Linear(5, 5),

            nn.ReLU(), # ReLU again

            # output layer produces 1 prediction
            nn.Linear(5, 1)
        )

    def forward(self, x):
        return self.layers(x) # pass the data through the layers

import torch.optim as optim

# Initialize the model, loss function, and optimizer
model_1000 = DiabetesModel1000()
criterion_1000 = nn.MSELoss() # MSE loss function
optimiser_1000 = optim.Adam(model_1000.parameters(), lr=0.001)

epochs = 1000 # 1000 epochs

model_1000.to(device)

for epoch in range(epochs):
    # use the train_loader to pass the inputs (X) and targets (y)
    for inputs, targets in train_loader:
        # pass to the GPU (hopefully)
        inputs, targets = inputs.to(device), targets.to(device)

        model_1000.train() # put the model object in train mode
        optimiser_1000.zero_grad() # reset the gradients
        outputs = model_1000(inputs) # create outputs
        loss = criterion_1000(outputs, targets) # compare with Y to get loss
        loss.backward() # backpropagate the loss
        optimiser_1000.step() # update the parameters based on this round of training

    # every 10 epochs we will print out the current loss
    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {round(loss.item(), 4)}')

# Evaluation (Exercise #1)
model_1000.eval() # testing mode
mse_values_1000 = [] # collect the MSE scores

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model_1000(inputs) # predict the test data

        # Calculate Mean Squared Error
        mse = criterion_1000(outputs, targets) # calculate mse for the batch
        mse_values_1000.append(mse.item()) # add to the list of MSE values

# Calculate and print the average MSE
avg_mse_1000 = np.mean(mse_values_1000)
print(f"Average MSE on test set (1000 epochs): {avg_mse_1000}")

# Compare model performance: 100 vs. 1000 epochs

mse_100 = 22161.48046875 # MSE from the original 100 epochs

comparison_ex1 = pd.DataFrame({
    'Epochs': [100, 1000],
    'Average Test MSE': [mse_100, avg_mse_1000]
})

print("Comparison of Model Performance:")
print(comparison_ex1.to_string(index=False))

# Calculate percentage improvement
improvement = ((mse_100 - avg_mse_1000) / mse_100) * 100

print(f"\nMSE reduction: {improvement:.2f}%")


Epoch [10/1000], Loss: 18026.2773
Epoch [20/1000], Loss: 19882.9512
Epoch [30/1000], Loss: 55676.2969
Epoch [40/1000], Loss: 23297.5156
Epoch [50/1000], Loss: 30337.9277
Epoch [60/1000], Loss: 46065.2969
Epoch [70/1000], Loss: 16350.4883
Epoch [80/1000], Loss: 50573.2266
Epoch [90/1000], Loss: 36609.9297
Epoch [100/1000], Loss: 31399.334
Epoch [110/1000], Loss: 8021.0254
Epoch [120/1000], Loss: 40556.3281
Epoch [130/1000], Loss: 24412.1387
Epoch [140/1000], Loss: 38209.293
Epoch [150/1000], Loss: 41697.7461
Epoch [160/1000], Loss: 31487.668
Epoch [170/1000], Loss: 25219.0312
Epoch [180/1000], Loss: 29804.9258
Epoch [190/1000], Loss: 60209.2109
Epoch [200/1000], Loss: 9354.8281
Epoch [210/1000], Loss: 18806.918
Epoch [220/1000], Loss: 47589.5078
Epoch [230/1000], Loss: 52338.4062
Epoch [240/1000], Loss: 2475.9324
Epoch [250/1000], Loss: 5089.4453
Epoch [260/1000], Loss: 34004.7578
Epoch [270/1000], Loss: 40135.0898
Epoch [280/1000], Loss: 47621.6953
Epoch [290/1000], Loss: 14500.8184
Ep

Increasing the number of epochs from 100 to 1,000 did not improve the model's performance in this experiment. The average test MSE increased from 22,161.48 to 23,895.69, representing a 7.83% increase in prediction error.

The training loss also continued to fluctuate, suggesting that the model had not fully stabilised. However, since the models were initialised separately, some of the difference may be due to random initialisation rather than the number of epochs alone.

Therefore, increasing the number of epochs does not necessarily guarantee better predictions.

In [13]:
## Exercise #2 – Comparing Neural Network Architectures

import torch
import torch.nn as nn

# Define the model
class DiabetesModelA(nn.Module):
    def __init__(self):
        super(DiabetesModelA, self).__init__()

        # we'll set up the layers as a sequence using nn.Sequential
        self.layers = nn.Sequential(

            # first layer has 10 neurons
            nn.Linear(10, 10),
            nn.ReLU(), # ReLU activation

            # second layer has 10 neurons
            nn.Linear(10, 10),
            nn.ReLU(), # ReLU again

            # output layer
            nn.Linear(10, 1)
        )

    def forward(self, x):
        return self.layers(x) # pass the data through the layers


import torch.optim as optim

# Initialize the model, loss function, and optimizer
model_A = DiabetesModelA()
criterion_A = nn.MSELoss() # MSE loss function
optimiser_A = optim.Adam(model_A.parameters(), lr=0.001)


# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Training loop
epochs = 1000 # 1000 epochs

model_A.to(device)

for epoch in range(epochs):
    # use the train_loader to pass the inputs (X) and targets (y)
    for inputs, targets in train_loader:
        # pass to the GPU (hopefully)
        inputs, targets = inputs.to(device), targets.to(device)

        model_A.train() # put the model object in train mode
        optimiser_A.zero_grad() # reset the gradients
        outputs = model_A(inputs) # create outputs
        loss = criterion_A(outputs, targets) # compare with Y to get loss
        loss.backward() # backpropagate the loss
        optimiser_A.step() # update the parameters

    # every 10 epochs we will print out the current loss
    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {round(loss.item(), 4)}')

# Evaluation (Model A)
model_A.eval() # testing mode
mse_values_A = [] # collect the MSE scores

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model_A(inputs) # predict the test data

        # Calculate Mean Squared Error
        mse = criterion_A(outputs, targets) # calculate mse for the batch
        mse_values_A.append(mse.item()) # add to the list of MSE values

# Calculate and print the average MSE
avg_mse_A = np.mean(mse_values_A)
print(f"Average MSE on test set (Model A): {avg_mse_A}")





Epoch [10/1000], Loss: 27610.9531
Epoch [20/1000], Loss: 24287.0371
Epoch [30/1000], Loss: 28230.3809
Epoch [40/1000], Loss: 21307.4551
Epoch [50/1000], Loss: 31448.3594
Epoch [60/1000], Loss: 65992.2344
Epoch [70/1000], Loss: 16844.3984
Epoch [80/1000], Loss: 16970.9141
Epoch [90/1000], Loss: 3796.4539
Epoch [100/1000], Loss: 15136.7422
Epoch [110/1000], Loss: 4783.7676
Epoch [120/1000], Loss: 2599.667
Epoch [130/1000], Loss: 681.2241
Epoch [140/1000], Loss: 5545.0889
Epoch [150/1000], Loss: 1050.8639
Epoch [160/1000], Loss: 6389.8066
Epoch [170/1000], Loss: 5981.5156
Epoch [180/1000], Loss: 1043.8303
Epoch [190/1000], Loss: 2171.8726
Epoch [200/1000], Loss: 955.6362
Epoch [210/1000], Loss: 7087.103
Epoch [220/1000], Loss: 1696.6512
Epoch [230/1000], Loss: 3114.0857
Epoch [240/1000], Loss: 2154.7739
Epoch [250/1000], Loss: 2569.6445
Epoch [260/1000], Loss: 3407.3416
Epoch [270/1000], Loss: 4079.6062
Epoch [280/1000], Loss: 1839.6439
Epoch [290/1000], Loss: 3657.6602
Epoch [300/1000], 

In [14]:
import torch
import torch.nn as nn

# Define the model
class DiabetesModelB(nn.Module):
    def __init__(self):
        super(DiabetesModelB, self).__init__()

        # we'll set up the layers as a sequence using nn.Sequential
        self.layers = nn.Sequential(

            # first layer has 20 neurons
            nn.Linear(10, 20),
            nn.ReLU(), # ReLU activation

            # second layer has 10 neurons
            nn.Linear(20, 10),
            nn.ReLU(), # ReLU again

            # output layer
            nn.Linear(10, 1)
        )

    def forward(self, x):
        return self.layers(x) # pass the data through the layers

import torch.optim as optim

# Initialize the model, loss function, and optimizer
model_B = DiabetesModelB()
criterion_B = nn.MSELoss() # MSE loss function
optimiser_B = optim.Adam(model_B.parameters(), lr=0.001)


# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Training loop
epochs = 1000 # 1000 epochs

model_B.to(device)

for epoch in range(epochs):
    # use the train_loader to pass the inputs (X) and targets (y)
    for inputs, targets in train_loader:
        # pass to the GPU (hopefully)
        inputs, targets = inputs.to(device), targets.to(device)

        model_B.train() # put the model object in train mode
        optimiser_B.zero_grad() # reset the gradients
        outputs = model_B(inputs) # create outputs
        loss = criterion_B(outputs, targets) # compare with Y to get loss
        loss.backward() # backpropagate the loss
        optimiser_B.step() # update the parameters

    # every 10 epochs we will print out the current loss
    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {round(loss.item(), 4)}')


# Evaluation (Model B)
model_B.eval() # testing mode
mse_values_B = [] # collect the MSE scores

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model_B(inputs) # predict the test data

        # Calculate Mean Squared Error
        mse = criterion_B(outputs, targets) # calculate mse for the batch
        mse_values_B.append(mse.item()) # add to the list of MSE values

# Calculate and print the average MSE
avg_mse_B = np.mean(mse_values_B)
print(f"Average MSE on test set (Model B): {avg_mse_B}")




Epoch [10/1000], Loss: 13740.7969
Epoch [20/1000], Loss: 37681.0938
Epoch [30/1000], Loss: 9264.7451
Epoch [40/1000], Loss: 72197.8984
Epoch [50/1000], Loss: 17703.5
Epoch [60/1000], Loss: 16100.7021
Epoch [70/1000], Loss: 2796.3162
Epoch [80/1000], Loss: 6065.0771
Epoch [90/1000], Loss: 1592.3719
Epoch [100/1000], Loss: 1465.6572
Epoch [110/1000], Loss: 9823.0176
Epoch [120/1000], Loss: 1979.7576
Epoch [130/1000], Loss: 10099.1689
Epoch [140/1000], Loss: 1326.1895
Epoch [150/1000], Loss: 1541.6365
Epoch [160/1000], Loss: 6573.6968
Epoch [170/1000], Loss: 1939.3094
Epoch [180/1000], Loss: 1710.1132
Epoch [190/1000], Loss: 2271.6387
Epoch [200/1000], Loss: 2510.6035
Epoch [210/1000], Loss: 2681.2788
Epoch [220/1000], Loss: 2931.6445
Epoch [230/1000], Loss: 9221.5889
Epoch [240/1000], Loss: 1379.2474
Epoch [250/1000], Loss: 4470.3867
Epoch [260/1000], Loss: 2246.3506
Epoch [270/1000], Loss: 1248.1206
Epoch [280/1000], Loss: 7742.2827
Epoch [290/1000], Loss: 467.7143
Epoch [300/1000], Los

In [15]:
import torch
import torch.nn as nn

# Define the model
class DiabetesModelC(nn.Module):
    def __init__(self):
        super(DiabetesModelC, self).__init__()

        # we'll set up the layers as a sequence using nn.Sequential
        self.layers = nn.Sequential(

            # first layer has 20 neurons
            nn.Linear(10, 20),
            nn.ReLU(), # ReLU activation

            # second layer has 10 neurons
            nn.Linear(20, 10),
            nn.ReLU(), # ReLU again

            # third layer has 5 neurons
            nn.Linear(10, 5),
            nn.ReLU(), # ReLU again

            # output layer
            nn.Linear(5, 1)
        )

    def forward(self, x):
        return self.layers(x) # pass the data through the layers


import torch.optim as optim

# Initialize the model, loss function, and optimizer
model_C = DiabetesModelC()
criterion_C = nn.MSELoss() # MSE loss function
optimiser_C = optim.Adam(model_C.parameters(), lr=0.001)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Training loop
epochs = 1000 # 1000 epochs

model_C.to(device)

for epoch in range(epochs):
    # use the train_loader to pass the inputs (X) and targets (y)
    for inputs, targets in train_loader:
        # pass to the GPU (hopefully)
        inputs, targets = inputs.to(device), targets.to(device)

        model_C.train() # put the model object in train mode
        optimiser_C.zero_grad() # reset the gradients
        outputs = model_C(inputs) # create outputs
        loss = criterion_C(outputs, targets) # compare with Y to get loss
        loss.backward() # backpropagate the loss
        optimiser_C.step() # update the parameters

    # every 10 epochs we will print out the current loss
    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {round(loss.item(), 4)}')


# Evaluation (Model C)
model_C.eval() # testing mode
mse_values_C = [] # collect the MSE scores

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model_C(inputs) # predict the test data

        # Calculate Mean Squared Error
        mse = criterion_C(outputs, targets) # calculate mse for the batch
        mse_values_C.append(mse.item()) # add to the list of MSE values

# Calculate and print the average MSE
avg_mse_C = np.mean(mse_values_C)
print(f"Average MSE on test set (Model C): {avg_mse_C}")


Epoch [10/1000], Loss: 30991.5371
Epoch [20/1000], Loss: 35562.6055
Epoch [30/1000], Loss: 26057.1367
Epoch [40/1000], Loss: 24852.459
Epoch [50/1000], Loss: 15128.4609
Epoch [60/1000], Loss: 2476.0552
Epoch [70/1000], Loss: 13885.0469
Epoch [80/1000], Loss: 4630.9189
Epoch [90/1000], Loss: 5620.4365
Epoch [100/1000], Loss: 241.7782
Epoch [110/1000], Loss: 5614.0054
Epoch [120/1000], Loss: 5004.3096
Epoch [130/1000], Loss: 3521.615
Epoch [140/1000], Loss: 1948.2522
Epoch [150/1000], Loss: 3009.8706
Epoch [160/1000], Loss: 7051.1069
Epoch [170/1000], Loss: 5774.4917
Epoch [180/1000], Loss: 4013.6245
Epoch [190/1000], Loss: 3134.8018
Epoch [200/1000], Loss: 4984.7334
Epoch [210/1000], Loss: 5057.2031
Epoch [220/1000], Loss: 3294.7188
Epoch [230/1000], Loss: 3351.3301
Epoch [240/1000], Loss: 4768.3975
Epoch [250/1000], Loss: 3712.6621
Epoch [260/1000], Loss: 3379.1841
Epoch [270/1000], Loss: 4715.3193
Epoch [280/1000], Loss: 1461.9603
Epoch [290/1000], Loss: 1649.4276
Epoch [300/1000], Lo

In [16]:
# Compare the performance of different neural network architectures

comparison = pd.DataFrame({
    'Model': ['Original', 'Model A', 'Model B', 'Model C'],
    'Architecture': ['10-5-5-1', '10-10-10-1', '10-20-10-1', '10-20-10-5-1'],
    'Average Test MSE': [2849.493896484375, avg_mse_A, avg_mse_B, avg_mse_C]
})

# Sort models by their MSE (lower is better)
comparison = comparison.sort_values(by='Average Test MSE')

print("Comparison of Neural Network Architectures:")
print(comparison.to_string(index=False))

# Identify the best performing model
print("\nBest performing model:")
print(comparison.iloc[0]['Model'])


Comparison of Neural Network Architectures:
   Model Architecture  Average Test MSE
 Model C 10-20-10-5-1       2842.563721
Original     10-5-5-1       2849.493896
 Model B   10-20-10-1       2866.029419
 Model A   10-10-10-1       2868.601685

Best performing model:
Model C


Exercise #2 – Results and Interpretation

It was tested three different neural network architectures by changing the number of neurons and hidden layers.

Model C (10-20-10-5-1) achieved the lowest test MSE of 2,842.56, slightly outperforming the original model (2,849.49). However, the differences between the four models were relatively small.

These results suggest that increasing the number of neurons or layers does not necessarily improve predictive performance.

Although Model C performed best in this experiment, we cannot conclude that it is the optimal architecture, as further testing and validation would be needed.